In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

from dynamicslib.consts import muEM
from dynamicslib.continuation import adaptive_cont
from dynamicslib.common import prop, jacobi_constant
from dynamicslib.targeter import dc_underconstrained
from tqdm.auto import tqdm
from dynamicslib.plotting import plotly_display, hodographs, test_new_family
from dynamicslib.common_targetters import spatial_perpendicular, xy_symmetric, planar_perpendicular, fullstate_minus_one, fullstate

In [ ]:
# targetter = xy_symmetric(1e-12, muEM)
targetter = spatial_perpendicular(1e-13, muEM)
# targetter = fullstate_minus_one(2, 0, 0.0, 1e-13, muEM)
# targetter = fullstate(1e-13, muEM)

func = targetter.f_df_stm

In [ ]:
plt.close("all")

x0 = np.array([0.990579721, 0.0, 0.126064556, 0, -0.019545496, 0])
tf = 0.859477746 * 5

# plt.close("all")
# %matplotlib ipympl
# xs = prop(x0, tf, muEM, int_tol=1e-14)
# f = plt.figure()
# # ax = f.add_subplot()
# # ax.plot(xs[1], xs[2])
# ax = f.add_subplot(projection='3d')
# ax.scatter(*xs[:3, [0, -1]])
# ax.set(xlabel='x',ylabel='y',zlabel='z')
# ax.plot(*xs[:3])
# ax.axis("equal")
# ax.grid(True)
# plt.show()

In [ ]:
X0 = targetter.get_X(x0, tf)
X1, df, stm = dc_underconstrained(
    X0, targetter.f_df_stm, 1e-10, fudge=0.5, max_step=1e-3, debug=True
)
svd = np.linalg.svd(df)
tangent = svd.Vh[-1]

In [ ]:
# %matplotlib ipympl
# x0, tf = targetter.get_x0(X1), targetter.get_period(X1)
# xs = prop(x0, tf, muEM, int_tol=1e-14)
# f = plt.figure()
# ax = f.add_subplot(projection="3d")
# ax.plot(*xs[:3])
# ax.axis("equal")
# plt.show()

In [ ]:
Xs1p, es1p, _ = adaptive_cont(
    X1,
    func,
    tangent,
    1e-4,
    1e-5,
    4.5,
    1e-12,
    rate=1.15,
    max_iter=9,
    target_iter=4,
    reduce_maxiter=1.5,
    exact_tangent=True,
)

In [ ]:
Xs1m, es1m, _ = adaptive_cont(
    X1,
    func,
    -tangent,
    1e-4,
    1e-5,
    2.5,
    1e-12,
    rate=1.15,
    max_iter=9,
    target_iter=4,
    reduce_maxiter=1.5,
    exact_tangent=True,
)

In [ ]:
Xs = [*Xs1p]
es = [*es1p]

param_names = [
    "Index",
    "Initial x",
    "Initial y",
    "Initial z",
    "Initial vx",
    "Initial vy",
    "Initial vz",
    "Period",
    "Jacobi Constant",
    "Stability Index",
    "Eig1",
    "Eig2",
    "Eig3",
    "Eig4",
    "Eig5",
    "Eig6",
]

xyzs = []
data = []


iterator = list(enumerate(zip(Xs, es)))
for i, (X, evals) in tqdm(iterator):
    x0 = targetter.get_x0(X).copy()
    tf = targetter.get_period(X)
    xyzs.append(prop(x0, tf, muEM, 1e-12))
    args = np.argsort(np.abs(evals))
    stabind = np.max([(np.abs(lam) + 1 / np.abs(lam)) / 2 for lam in evals])
    jc = jacobi_constant(x0)
    data.append([i, *x0, tf, jc, stabind, *evals])

In [ ]:
df = pd.DataFrame(data, columns=param_names).set_index("Index")
for key in param_names[1:]:
    if np.all(df[key].abs() < 1e-8):
        df = df.drop(columns=[key])

df.to_csv("new_test.csv")

In [ ]:
test_new_family(df)

In [ ]:
a = None
b = None
c = 10
plotly_display(xyzs[a:b:c], df[a:b:c], port=8000)